# Serve Gemma 2B with vLLM & LoRA on Google Colab (Free T4)

This notebook demonstrates how to serve your fine-tuned `google/gemma-2-2b-it` model with the `sachin19566/gemma-2b-hindi-lora` adapter using **vLLM** on a free **T4 GPU** in Google Colab.

### Important Checklist:
1. **Enable T4 GPU**: Go to **Runtime** -> **Change runtime type** -> select **T4 GPU** under Hardware Accelerator. Click Save.
2. **Restart Session**: After running the `pip install` cell, you **MUST** click **Runtime** -> **Restart session** (or `Ctrl+M .`). This ensures that Google Colab correctly links the CUDA drivers to the newly installed vLLM binaries.

### Step 1: Install vLLM and Outlines
We first uninstall `torchaudio` to bypass the PyTorch/TorchAudio CUDA compilation version mismatch, as audio processing libraries are not needed for text-only LLM serving.

In [ ]:
# Uninstall mismatched audio package to avoid PyTorch CUDA conflict
!pip uninstall -y torchaudio

# Install vllm and outlines
!pip install -q vllm outlines

### Step 2: Verify GPU / CUDA Environment
If this cell prints `False` or doesn't detect a GPU, vLLM will raise a `RuntimeError: Failed to infer device type`. Ensure GPU is enabled in your Colab settings.

In [ ]:
import torch
import os

cuda_available = torch.cuda.is_available()
print("Is CUDA (GPU) available in PyTorch?:", cuda_available)
if cuda_available:
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("PyTorch CUDA Version:", torch.version.cuda)
    !nvidia-smi
else:
    print("\n*** ERROR: GPU accelerator is NOT active! ***")
    print("Go to Runtime -> Change runtime type -> select T4 GPU under Hardware accelerator, and run this notebook again.\n")

### Step 3: Set Hugging Face Access Token & Engine Options

In [ ]:
from google.colab import userdata
import os

# Set Hugging Face Token
if "HF_TOKEN" not in os.environ:
    try:
        os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("Enter your Hugging Face Access Token: ")

# Enable verbose debug logging for vLLM
os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"

### Step 4: Launch vLLM Server in the Background
We start the vLLM server, redirect output to `vllm.log`, and monitor the logs as the model downloads and loads in the background.

In [ ]:
import subprocess
import time

print("Starting vLLM server in the background...")

# T4 GPU & Gemma 2 Compatibility Workaround:
# 1. Gemma 2 does not support float16 (blocked due to numerical instability errors).
# 2. Tesla T4 GPU does not support bfloat16 (compute capability 7.5; requires 8.0+).
# 3. Solution: Use float32 precision (--dtype float32). Gemma 2B in FP32 takes ~10.4GB VRAM.
# 4. Memory Allocations:
#    - gpu-memory-utilization 0.85: pre-allocates 12.75GB on the 15GB T4 GPU. Leaving ~2.3GB for KV Cache.
#    - max-model-len 512: limits KV cache size to fit in the remaining 2.3GB without OOM.
vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "google/gemma-2-2b-it",
    "--enable-lora",
    "--lora-modules", "gemma-hindi-lora=sachin19566/gemma-2b-hindi-lora",
    "--max-lora-rank", "16",
    "--gpu-memory-utilization", "0.85",
    "--max-model-len", "512",
    "--dtype", "float32",
    "--enforce-eager"
]

# Start process
with open("vllm.log", "w") as log_file:
    process = subprocess.Popen(vllm_cmd, stdout=log_file, stderr=log_file)

print(f"vLLM server started with PID: {process.pid}")
print("Waiting 90 seconds for model to load. Monitoring log file...")

# Poll logs periodically to show startup progress
for i in range(12):
    time.sleep(10)
    print(f"[{i*10 + 10}s elapsed] Monitoring logs...")
    if os.path.exists("vllm.log"):
        with open("vllm.log", "r") as f:
            logs = f.readlines()
            # Check if server has started
            started = any("Uvicorn running on" in line or "Application startup complete" in line for line in logs)
            if started:
                print("\nSuccess! vLLM Server is now running and ready to receive requests.")
                break
            else:               
                # Print the last 4 log lines
                print("".join(logs[-4:]))

# Final log printout (last 80 lines to show full traceback if it crashed)
print("\n--- Final lines of vllm.log (Traceback debugging) ---")
with open("vllm.log", "r") as f:
    print("".join(f.readlines()[-80:]))

### Step 5: Query the Model with 300-Word Regex Constraint

In [ ]:
from huggingface_hub import hf_hub_download
import json
import requests
import os

repo_id = "sachin19566/gemma-2b-hindi-lora"

# 1. Download vocabulary file
print("Downloading vocabulary file...")
vocab_path = hf_hub_download(repo_id=repo_id, filename="hindi_vocab.json")
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = json.load(f)
print(f"Loaded {len(vocab)} allowed words.")

# 2. Build guided regex pattern
word_choices = "|".join(vocab)
regex_pattern = f"^({word_choices}|[।?,!\\s\\n])+"

# 3. Call local vLLM API endpoint
url = "http://localhost:8000/v1/chat/completions"
headers = {"Content-Type": "application/json"}
data = {
    "model": "gemma-hindi-lora",
    "messages": [
        {"role": "user", "content": "नमस्ते दोस्त, आप कैसे हैं और क्या खाना खा रहे हैं?"}
    ],
    "guided_regex": regex_pattern,
    "temperature": 0.1,
    "max_tokens": 64
}

print("Sending request to vLLM server...")
try:
    response = requests.post(url, headers=headers, json=data)
    response_json = response.json()
    print("\n--- Response from vLLM ---")
    print("User:", data["messages"][0]["content"])
    print("Model:", response_json["choices"][0]["message"]["content"])
except Exception as e:
    print("Failed to query server. Make sure the server has loaded completely.")
    print("Error details:", e)
    # Print log file output for debugging
    if os.path.exists("vllm.log"):
        print("\n--- Debugging: Last 50 lines of vllm.log ---")
        with open("vllm.log", "r") as f:
            print("").join(f.readlines()[-50:]))

### Step 6: Shut Down the Server

In [ ]:
try:
    process.terminate()
    print("Sent termination signal to vLLM process.")
    process.wait(timeout=10)
    print("vLLM server stopped.")
except Exception as e:
    print("Error stopping process:", e)